# Optimise a control pulse from a saved `.jld2` run

Loads a trajectory written by `run_sim_1st_order` / `save_run_data`, reconciles it against this package's CPU 1st-order physics, then runs the differentiable composite-pulse optimiser in `src/pulse_optimizer.jl` (via [`optimise_control_pulse_from_jld2`](../src/jld2_pulse_loader.jl)).

Edit the **settings** cell, then run all cells. The last cell overlays the **seed** control pulse (hop-0 `initial_guess`) against the **optimised** control pulse.

**Kernel:** Julia, with this repo as the active project (`Pkg.activate` in the next cell). `IJulia` must be installed in that environment (or in a shared Julia depot that this kernel can see).


In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))

using InhomogeneousSpinCavityDynamics
using Plots

gr()
default(size=(900, 400), legend=:topright)


## Settings

- `JLD2_PATH` — source run (must contain `SIM_SETTING`, `SYSTEM_CONFIG`, `PULSE_CONFIG`, and the saved trajectory).
- `USE_SIGNAL` — if `true`, the file's leading signal pulse(s) stay on as a **fixed** background drive (never part of `u`). If `false`, that drive is identically zero.
- `SAVE_LOG` — if `true`, writes `<basename>_optrunlog.jld2` and `<basename>_opt_pulsemat.csv` (see `save_optimisation_run_log`).

`num_epochs` / `n_hops` default to the package's modest smoke-test budget. Raise them for a real search.


In [ ]:
# --- source run ---
JLD2_PATH = joinpath(@__DIR__, "..", "data", "run_001.jld2")

# --- signal / logging ---
USE_SIGNAL = true          # USE_SIGNAL mode flag (use_signal=)
SAVE_LOG   = true          # write optrunlog.jld2 + opt_pulsemat.csv
LOG_OUT_DIR = nothing      # nothing = alongside JLD2_PATH; or a directory string

# --- PULSE_CONFIG split ---
N_SIGNAL = 1               # leading entries treated as the fixed signal

# --- composite control pulse (B-spline) ---
K         = 2              # number of sub-pulses
N_COEFF_A = 6              # amplitude spline coefficients per sub-pulse
N_COEFF_F = 4              # frequency spline coefficients per sub-pulse

# --- optimiser (ForwardDiff / Adam + basin-hopping) ---
SEED           = 42
NUM_EPOCHS     = 30
N_HOPS         = 3
LEARNING_RATE  = 0.05
PATIENCE       = 5
HOP_PATIENCE   = 2
TOL            = 1e-3
LAMBDA_AREA    = 100.0

# --- reconciliation gate (refuses to optimise if the CPU re-solve mismatches the file) ---
RECONCILE  = true
RTOL_CHECK = 1e-3

isfile(JLD2_PATH) || error("JLD2 file not found: $JLD2_PATH")
println("JLD2_PATH  = $JLD2_PATH")
println("USE_SIGNAL = $USE_SIGNAL")
println("SAVE_LOG   = $SAVE_LOG")


## Inspect the loaded run


In [ ]:
data = load_jld2_run(JLD2_PATH)
signal_cfg, control_cfg = split_signal_control(data.PULSE_CONFIG; n_signal=N_SIGNAL)

println("SIM_SETTING.Ttotal  = $(data.SIM_SETTING.Ttotal)")
println("SIM_SETTING.M_delta = $(data.SIM_SETTING.M_delta)")
println("n pulses in PULSE_CONFIG = $(length(data.PULSE_CONFIG))")
println("signal pulses  ($(length(signal_cfg))):", [p.name for p in signal_cfg])
println("control pulses ($(length(control_cfg))):", [p.name for p in control_cfg])


## Run the optimiser

This calls `optimise_control_pulse_from_jld2`, which uses `run_sim_1st_order_pure` / `optimise_composite_pulse` from `src/pulse_optimizer.jl`.
Each epoch differentiates through a full 1st-order solve — this can take a long time on a fine ensemble.


In [ ]:
best_u, best_cost, pulse, signal_E_of_t, d, data = optimise_control_pulse_from_jld2(
    JLD2_PATH, K, N_COEFF_A, N_COEFF_F;
    n_signal   = N_SIGNAL,
    use_signal = USE_SIGNAL,
    reconcile  = RECONCILE,
    rtol_check = RTOL_CHECK,
    save_log   = SAVE_LOG,
    log_out_dir = LOG_OUT_DIR,
    seed          = SEED,
    num_epochs    = NUM_EPOCHS,
    n_hops        = N_HOPS,
    learning_rate = LEARNING_RATE,
    patience      = PATIENCE,
    hop_patience  = HOP_PATIENCE,
    tol           = TOL,
    lambda_area   = LAMBDA_AREA,
)

u_seed = initial_guess(pulse; seed=SEED)
seed_metrics = pulse_cost(u_seed, pulse, d; signal_E_of_t=signal_E_of_t, lambda_area=LAMBDA_AREA)
final_metrics = pulse_cost(best_u, pulse, d; signal_E_of_t=signal_E_of_t, lambda_area=LAMBDA_AREA)

println()
println("USE_SIGNAL = $USE_SIGNAL   SAVE_LOG = $SAVE_LOG")
println("seed  cost=$(round(seed_metrics[1]; digits=4))  inversion=$(round(seed_metrics[2]; digits=4))  coherence=$(round(seed_metrics[3]; digits=4))  area=$(round(seed_metrics[5]; digits=4))")
println("opt   cost=$(round(final_metrics[1]; digits=4))  inversion=$(round(final_metrics[2]; digits=4))  coherence=$(round(final_metrics[3]; digits=4))  area=$(round(final_metrics[5]; digits=4))")
println("best_cost returned = $(round(best_cost; digits=4))")

if SAVE_LOG
    optrunlog_path, pulsemat_path = InhomogeneousSpinCavityDynamics.optrunlog_paths(
        JLD2_PATH; out_dir=LOG_OUT_DIR,
    )
    println("optrunlog : $optrunlog_path")
    println("opt pulse : $pulsemat_path")
end


## Plot seed vs optimised control pulse

These are **control-only** drives (`build_E_of_t(pulse, u)`), not combined with the signal. That matches what `save_optimisation_run_log` writes to `*_opt_pulsemat.csv`.


In [ ]:
N_PLOT = max(data.SIM_SETTING.Nt_save, 2000)
t_end  = pulse.T_max
t_us   = collect(range(0.0, t_end; length=N_PLOT)) .* 1e6

E_seed = build_E_of_t(pulse, u_seed)
E_opt  = build_E_of_t(pulse, best_u)

Ex_s, Ep_s = sample_E_of_t(E_seed, t_end, N_PLOT)
Ex_o, Ep_o = sample_E_of_t(E_opt,  t_end, N_PLOT)
abs_s = hypot.(Ex_s, Ep_s)
abs_o = hypot.(Ex_o, Ep_o)

plt_abs = plot(
    t_us, abs_s;
    label="seed |E|", xlabel="time (μs)", ylabel="Amplitude",
    title="Control pulse: seed vs optimised",
    linestyle=:dash,
)
plot!(plt_abs, t_us, abs_o; label="optimised |E|")
display(plt_abs)

plt_re = plot(
    t_us, Ex_s;
    label="seed Re[E]", xlabel="time (μs)", ylabel="Re[E(t)]",
    title="Control pulse real part", linestyle=:dash,
)
plot!(plt_re, t_us, Ex_o; label="optimised Re[E]")
display(plt_re)

plt_im = plot(
    t_us, Ep_s;
    label="seed Im[E]", xlabel="time (μs)", ylabel="Im[E(t)]",
    title="Control pulse imag part", linestyle=:dash,
)
plot!(plt_im, t_us, Ep_o; label="optimised Im[E]")
display(plt_im)
